In [1]:
import json
import pandas as pd
import numpy as np

from pathlib import Path

ModuleNotFoundError: No module named 'pandas'

In [ ]:
data_folder = Path("data")

json_files = sorted(data_folder.glob("*.json"))

print("Total game files:", len(json_files))

Total game files: 100


In [ ]:
with open(json_files[0], "r") as f:
    game = json.load(f)

print(game.keys())

dict_keys(['schema_version', 'game', 'map', 'board', 'colors', 'player_state', 'buildings_by_color', 'resource_freqdeck', 'development_listdeck', 'random_state', 'action_records', 'num_turns', 'current_player_index', 'current_turn_index', 'current_prompt', 'is_initial_build_phase', 'is_discarding', 'discard_counts', 'is_moving_knight', 'is_road_building', 'free_roads_available', 'is_resolving_trade', 'current_trade', 'acceptees'])


In [ ]:
for record in game["action_records"][:16]:
    print(record[0])

['ORANGE', 'BUILD_SETTLEMENT', 21]
['ORANGE', 'BUILD_ROAD', [16, 21]]
['RED', 'BUILD_SETTLEMENT', 4]
['RED', 'BUILD_ROAD', [4, 15]]
['WHITE', 'BUILD_SETTLEMENT', 45]
['WHITE', 'BUILD_ROAD', [45, 47]]
['BLUE', 'BUILD_SETTLEMENT', 7]
['BLUE', 'BUILD_ROAD', [7, 24]]
['BLUE', 'BUILD_SETTLEMENT', 39]
['BLUE', 'BUILD_ROAD', [39, 41]]
['WHITE', 'BUILD_SETTLEMENT', 29]
['WHITE', 'BUILD_ROAD', [10, 29]]
['RED', 'BUILD_SETTLEMENT', 25]
['RED', 'BUILD_ROAD', [25, 26]]
['ORANGE', 'BUILD_SETTLEMENT', 37]
['ORANGE', 'BUILD_ROAD', [14, 37]]


In [ ]:
def extract_initial_placements(game):

    game_id = game["game"]["id"]
    colors = game["colors"]

    players = {}

    # Initialize player records
    for order, color in enumerate(colors, start=1):

        players[color] = {
            "Game_ID": game_id,
            "Player": color,
            "Placement_Order": order,
            "Settlement_1": None,
            "Road_1": None,
            "Settlement_2": None,
            "Road_2": None
        }

    # Only extract the first 16 actions
    actions = [
        record[0]
        for record in game["action_records"][:16]
    ]

    for color, action_type, value in actions:

        player = players[color]

        if action_type == "BUILD_SETTLEMENT":

            if player["Settlement_1"] is None:
                player["Settlement_1"] = value

            else:
                player["Settlement_2"] = value

        elif action_type == "BUILD_ROAD":

            if player["Road_1"] is None:
                player["Road_1"] = value

            else:
                player["Road_2"] = value

        else:
            raise ValueError(
                f"Unexpected setup action: {action_type}"
            )

    # Return four player records
    return list(players.values())

In [ ]:
records = extract_initial_placements(game)

df_test = pd.DataFrame(records)

display(df_test)

,Game_ID,Player,Placement_Order,Settlement_1,Road_1,Settlement_2,Road_2
0,0049a458-311f-42c8-9b38-2d675e845336,ORANGE,1,21,"[16, 21]",37,"[14, 37]"
1,0049a458-311f-42c8-9b38-2d675e845336,RED,2,4,"[4, 15]",25,"[25, 26]"
2,0049a458-311f-42c8-9b38-2d675e845336,WHITE,3,45,"[45, 47]",29,"[10, 29]"
3,0049a458-311f-42c8-9b38-2d675e845336,BLUE,4,7,"[7, 24]",39,"[39, 41]"


In [ ]:
all_records = []

for file in json_files:

    with open(file, "r") as f:
        game = json.load(f)

    records = extract_initial_placements(game)

    all_records.extend(records)

df = pd.DataFrame(all_records)

display(df.head(10))

,Game_ID,Player,Placement_Order,Settlement_1,Road_1,Settlement_2,Road_2
0,0049a458-311f-42c8-9b38-2d675e845336,ORANGE,1,21,"[16, 21]",37,"[14, 37]"
1,0049a458-311f-42c8-9b38-2d675e845336,RED,2,4,"[4, 15]",25,"[25, 26]"
2,0049a458-311f-42c8-9b38-2d675e845336,WHITE,3,45,"[45, 47]",29,"[10, 29]"
3,0049a458-311f-42c8-9b38-2d675e845336,BLUE,4,7,"[7, 24]",39,"[39, 41]"
4,05efc4b3-9a75-47e6-ad4c-f6c3de4a8c2d,RED,1,9,"[8, 9]",3,"[3, 12]"
5,05efc4b3-9a75-47e6-ad4c-f6c3de4a8c2d,ORANGE,2,6,"[6, 7]",21,"[19, 21]"
6,05efc4b3-9a75-47e6-ad4c-f6c3de4a8c2d,BLUE,3,14,"[14, 15]",17,"[17, 18]"
7,05efc4b3-9a75-47e6-ad4c-f6c3de4a8c2d,WHITE,4,38,"[37, 38]",25,"[25, 26]"
8,08d13ff8-11d6-4991-b43d-ce6589d26d50,BLUE,1,11,"[11, 32]",4,"[3, 4]"
9,08d13ff8-11d6-4991-b43d-ce6589d26d50,WHITE,2,21,"[19, 21]",50,"[49, 50]"


In [ ]:
print("Total games:", df["Game_ID"].nunique())

print("Total records:", len(df))

print("Dataset shape:", df.shape)

Total games: 100
Total records: 400
Dataset shape: (400, 7)


In [ ]:
df.isna().sum()

Game_ID            0
Player             0
Placement_Order    0
Settlement_1       0
Road_1             0
Settlement_2       0
Road_2             0
dtype: int64

In [ ]:
df.duplicated(
    subset=["Game_ID", "Player"]
).sum()

np.int64(0)

In [ ]:
df.groupby("Game_ID").size().value_counts()

4    100
Name: count, dtype: int64

In [ ]:
for i in [1, 2]:

    valid = df.apply(
        lambda row: row[f"Settlement_{i}"]
        in row[f"Road_{i}"],
        axis=1
    )

    print(f"Round {i} invalid roads:", (~valid).sum())

Round 1 invalid roads: 0
Round 2 invalid roads: 0


In [ ]:
map_records = []

for file in json_files:

    with open(file, "r") as f:
        game = json.load(f)

    game_id = game["game"]["id"]

    for tile in game["map"]["tiles"]:

        if tile["type"] == "LAND":

            map_records.append({
                "Game_ID": game_id,
                "Tile_ID": tile["id"],
                "Coordinate_X": tile["coordinate"][0],
                "Coordinate_Y": tile["coordinate"][1],
                "Coordinate_Z": tile["coordinate"][2],
                "Resource": tile["resource"],
                "Number": tile["number"]
            })

df_map = pd.DataFrame(map_records)

display(df_map.head(10))

,Game_ID,Tile_ID,Coordinate_X,Coordinate_Y,Coordinate_Z,Resource,Number
0,0049a458-311f-42c8-9b38-2d675e845336,0,0,0,0,WHEAT,11.0
1,0049a458-311f-42c8-9b38-2d675e845336,1,1,-1,0,BRICK,10.0
2,0049a458-311f-42c8-9b38-2d675e845336,2,0,-1,1,SHEEP,3.0
3,0049a458-311f-42c8-9b38-2d675e845336,3,-1,0,1,WHEAT,6.0
4,0049a458-311f-42c8-9b38-2d675e845336,4,-1,1,0,ORE,5.0
5,0049a458-311f-42c8-9b38-2d675e845336,5,0,1,-1,BRICK,4.0
6,0049a458-311f-42c8-9b38-2d675e845336,6,1,0,-1,WHEAT,9.0
7,0049a458-311f-42c8-9b38-2d675e845336,7,2,-2,0,SHEEP,5.0
8,0049a458-311f-42c8-9b38-2d675e845336,8,1,-2,1,SHEEP,8.0
9,0049a458-311f-42c8-9b38-2d675e845336,9,0,-2,2,WHEAT,4.0


In [ ]:
output_folder = Path("processed_data")

output_folder.mkdir(exist_ok=True)

df.to_csv(
    output_folder / "initial_placements.csv",
    index=False
)

df_map.to_csv(
    output_folder / "map_tiles.csv",
    index=False
)